In [1]:
import duckdb
from pathlib import Path

DB_PATH = "developer_project.duckdb"
PARQUET_DIR = Path("outputs/v11_cluster_exports")

con = duckdb.connect(DB_PATH)

print("Connected to:", DB_PATH)
print("Parquet folder:", PARQUET_DIR.resolve())

Connected to: developer_project.duckdb
Parquet folder: /Users/pmazolew/Documents/GitHub/Spring2026_IndustryProject/outputs/v11_cluster_exports


In [2]:
parquet_files = sorted(PARQUET_DIR.glob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(f"No parquet files found in {PARQUET_DIR.resolve()}")

for f in parquet_files:
    print(f.name)

dev_gmm_weekly_clusters_v1.parquet
dev_lifecycle_cluster_membership_v11_active.parquet
dev_lifecycle_cluster_membership_v11_at_risk.parquet
dev_lifecycle_cluster_membership_v11_combined.parquet
dev_lifecycle_cluster_membership_v11_cooling.parquet
dev_lifecycle_cluster_membership_v11_dormant.parquet
dev_lifecycle_cluster_membership_v11_final.parquet
dev_lifecycle_cluster_membership_v11_profile_summary.parquet
dev_lifecycle_cluster_membership_v11_unactivated.parquet
dev_lifecycle_cluster_run_stats_v11.parquet


In [3]:
v11_tables = [
    "dev_lifecycle_cluster_membership_v11_active",
    "dev_lifecycle_cluster_membership_v11_cooling",
    "dev_lifecycle_cluster_membership_v11_at_risk",
    "dev_lifecycle_cluster_membership_v11_dormant",
    "dev_lifecycle_cluster_membership_v11_unactivated",
    "dev_lifecycle_cluster_membership_v11_combined",
    "dev_lifecycle_cluster_membership_v11_final",
    "dev_lifecycle_cluster_membership_v11_profile_summary",
    "dev_lifecycle_cluster_run_stats_v11"
]

for table in v11_tables:
    con.execute(f"DROP TABLE IF EXISTS {table}")

print("Dropped existing/partial V11 clustering tables.")

Dropped existing/partial V11 clustering tables.


In [4]:
table_files = {
    "dev_lifecycle_cluster_membership_v11_active": "dev_lifecycle_cluster_membership_v11_active.parquet",
    "dev_lifecycle_cluster_membership_v11_cooling": "dev_lifecycle_cluster_membership_v11_cooling.parquet",
    "dev_lifecycle_cluster_membership_v11_at_risk": "dev_lifecycle_cluster_membership_v11_at_risk.parquet",
    "dev_lifecycle_cluster_membership_v11_dormant": "dev_lifecycle_cluster_membership_v11_dormant.parquet",
    "dev_lifecycle_cluster_membership_v11_unactivated": "dev_lifecycle_cluster_membership_v11_unactivated.parquet",
    "dev_lifecycle_cluster_membership_v11_combined": "dev_lifecycle_cluster_membership_v11_combined.parquet",
    "dev_lifecycle_cluster_membership_v11_final": "dev_lifecycle_cluster_membership_v11_final.parquet",
    "dev_lifecycle_cluster_membership_v11_profile_summary": "dev_lifecycle_cluster_membership_v11_profile_summary.parquet",
    "dev_lifecycle_cluster_run_stats_v11": "dev_lifecycle_cluster_run_stats_v11.parquet",
}

for table_name, file_name in table_files.items():
    file_path = PARQUET_DIR / file_name
    
    if file_path.exists():
        print(f"Loading {file_name} into {table_name}...")
        con.execute(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *
            FROM read_parquet('{file_path.as_posix()}')
        """)
    else:
        print(f"Skipping missing file: {file_name}")

print("Finished loading available V11 Parquet files into DuckDB.")

Loading dev_lifecycle_cluster_membership_v11_active.parquet into dev_lifecycle_cluster_membership_v11_active...
Loading dev_lifecycle_cluster_membership_v11_cooling.parquet into dev_lifecycle_cluster_membership_v11_cooling...
Loading dev_lifecycle_cluster_membership_v11_at_risk.parquet into dev_lifecycle_cluster_membership_v11_at_risk...
Loading dev_lifecycle_cluster_membership_v11_dormant.parquet into dev_lifecycle_cluster_membership_v11_dormant...
Loading dev_lifecycle_cluster_membership_v11_unactivated.parquet into dev_lifecycle_cluster_membership_v11_unactivated...
Loading dev_lifecycle_cluster_membership_v11_combined.parquet into dev_lifecycle_cluster_membership_v11_combined...
Loading dev_lifecycle_cluster_membership_v11_final.parquet into dev_lifecycle_cluster_membership_v11_final...
Loading dev_lifecycle_cluster_membership_v11_profile_summary.parquet into dev_lifecycle_cluster_membership_v11_profile_summary...
Loading dev_lifecycle_cluster_run_stats_v11.parquet into dev_lifecyc

In [5]:
for table in v11_tables:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {n:,} rows")
    except Exception as e:
        print(f"{table}: not found")

dev_lifecycle_cluster_membership_v11_active: 418,049 rows
dev_lifecycle_cluster_membership_v11_cooling: 356,500 rows
dev_lifecycle_cluster_membership_v11_at_risk: 1,580,877 rows
dev_lifecycle_cluster_membership_v11_dormant: 5,304,852 rows
dev_lifecycle_cluster_membership_v11_unactivated: 1,721,230 rows
dev_lifecycle_cluster_membership_v11_combined: 9,381,508 rows
dev_lifecycle_cluster_membership_v11_final: 9,381,508 rows
dev_lifecycle_cluster_membership_v11_profile_summary: 26 rows
dev_lifecycle_cluster_run_stats_v11: 4 rows


In [6]:
con.execute("""
SELECT
    stratum,
    COUNT(*) AS n_developers,
    COUNT(DISTINCT cluster_key) AS n_cluster_keys,
    AVG(cluster_probability) AS avg_cluster_probability,
    AVG(outlier_score) AS avg_outlier_score
FROM dev_lifecycle_cluster_membership_v11_final
GROUP BY stratum
ORDER BY n_developers DESC
""").df()

,stratum,n_developers,n_cluster_keys,avg_cluster_probability,avg_outlier_score
0,dormant,5304852,3,1.000000,0.000000
1,unactivated,1721230,1,1.000000,0.000000
2,at_risk,1580877,7,0.753555,0.124177
3,active,418049,7,0.655923,0.109012
4,cooling,356500,8,0.709506,0.107562


In [7]:
con.execute("""
SELECT *
FROM dev_lifecycle_cluster_run_stats_v11
ORDER BY stratum
""").df()

,stratum,method,n_rows,n_features,features,n_clusters_excluding_noise,noise_count,noise_pct,min_cluster_size,min_samples,cluster_selection_epsilon
0,active,hdbscan,418049,12,"log_activity_count_0_30d, log_activity_count_3...",6,105723,0.252896,15000,300,0.08
1,at_risk,hdbscan,1580877,7,"log_activity_count_30_90d, log_activity_count_...",6,97500,0.061675,90000,1200,0.18
2,cooling,hdbscan,356500,8,"log_activity_count_30_90d, log_activity_count_...",7,81368,0.228241,18000,350,0.10
3,dormant,rule_based_segmentation,5304852,5,"developer_effort_score, build_share_lifetime, ...",3,0,0.000000,<NA>,<NA>,NaN


In [8]:
# ============================================================
# Load Weekly GMM Cluster Output
# ============================================================

GMM_WEEKLY_TABLE = "dev_gmm_weekly_clusters_v1"
GMM_WEEKLY_FILE = PARQUET_DIR / "dev_gmm_weekly_clusters_v1.parquet"

if GMM_WEEKLY_FILE.exists():
    print(f"Loading {GMM_WEEKLY_FILE.name} into {GMM_WEEKLY_TABLE}...")

    con.execute(f"""
        CREATE OR REPLACE TABLE {GMM_WEEKLY_TABLE} AS
        SELECT *
        FROM read_parquet('{GMM_WEEKLY_FILE.as_posix()}')
    """)

    print(f"Loaded {GMM_WEEKLY_TABLE} successfully.")
else:
    print(f"Missing file: {GMM_WEEKLY_FILE}")

Loading dev_gmm_weekly_clusters_v1.parquet into dev_gmm_weekly_clusters_v1...
Loaded dev_gmm_weekly_clusters_v1 successfully.


In [9]:
# Check row counts and date range
con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT developer_id) AS n_developers,
    MIN(week_start) AS min_week,
    MAX(week_start) AS max_week,
    COUNT(DISTINCT gmm_weekly_cluster_id) AS n_gmm_clusters
FROM dev_gmm_weekly_clusters_v1
""").df()

,n_rows,n_developers,min_week,max_week,n_gmm_clusters
0,14908480,7660278,2019-12-30,2026-03-09,3


In [10]:
con.execute("""
SELECT *
FROM dev_gmm_weekly_clusters_v1
LIMIT 10
""").df()

,developer_id,week_start,gmm_weekly_cluster_id,gmm_weekly_max_posterior,gmm_weekly_prob_c0,gmm_weekly_prob_c1,gmm_weekly_prob_c2
0,2002098,2025-08-25,0,0.999995,0.999995,4.123108e-12,5.373117e-06
1,9923949,2025-12-15,2,1.000000,0.000000,2.950233e-10,1.000000e+00
2,9929467,2025-12-15,2,1.000000,0.000000,2.950233e-10,1.000000e+00
3,3455258,2024-03-18,2,1.000000,0.000000,2.845153e-07,9.999997e-01
4,6514032,2025-03-17,2,1.000000,0.000000,7.210809e-09,1.000000e+00
5,5609680,2023-03-20,2,1.000000,0.000000,1.115436e-08,1.000000e+00
6,6068822,2021-04-12,2,1.000000,0.000000,1.933128e-07,9.999998e-01
7,5844153,2024-03-18,2,1.000000,0.000000,1.656332e-09,1.000000e+00
8,3497619,2022-03-21,2,1.000000,0.000000,7.425524e-09,1.000000e+00
9,1937880,2021-04-19,0,1.000000,1.000000,8.742172e-13,9.146222e-09


In [11]:
con.execute("""
SELECT
    gmm_weekly_cluster_id,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT developer_id) AS n_developers,
    AVG(gmm_weekly_max_posterior) AS avg_max_posterior
FROM dev_gmm_weekly_clusters_v1
GROUP BY gmm_weekly_cluster_id
ORDER BY gmm_weekly_cluster_id
""").df()

,gmm_weekly_cluster_id,n_rows,n_developers,avg_max_posterior
0,0,8992453,4302668,0.999994
1,1,382890,177375,0.996337
2,2,5533137,4965049,0.999726


## Load HMM parquet exports

Copy HMM parquet files from Dropbox (or teammate exports) into **`toexport_HMM/`** at the repo root:

```text
Spring2026_IndustryProject/
  toexport_HMM/
    dev_hmm_weekly_states_v1.parquet
    dev_hmm_state_profile_v1.parquet
    dev_hmm_developer_journey_v1.parquet
    dev_hmm_transition_matrix_v1.parquet
    dev_hmm_run_stats_v1.parquet
```

**Optional — analysis v2 tables** (from [`NVIDIA_HMM_Journey_Analysis_v2.ipynb`](NVIDIA_HMM_Journey_Analysis_v2.ipynb), if you have them):

```text
    dev_hmm_state_labels_v2.parquet
    dev_hmm_hdbscan_alignment_v2.parquet
    dev_hmm_lifecycle_signal_summary_v2.parquet
    dev_hmm_hdbscan_alignment_summary_v2.parquet
    dev_hmm_journey_archetype_summary_v2.parquet
    dev_hmm_business_signals_v2.parquet
```

All files can live in the same `toexport_HMM/` folder. Missing files are skipped.

In [12]:
HMM_PARQUET_DIR = Path("toexport_HMM")
HMM_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

hmm_v1_tables = [
    "dev_hmm_weekly_states_v1",
    "dev_hmm_state_profile_v1",
    "dev_hmm_developer_journey_v1",
    "dev_hmm_transition_matrix_v1",
    "dev_hmm_run_stats_v1",
]

hmm_v2_tables = [
    "dev_hmm_state_labels_v2",
    "dev_hmm_hdbscan_alignment_v2",
    "dev_hmm_lifecycle_signal_summary_v2",
    "dev_hmm_hdbscan_alignment_summary_v2",
    "dev_hmm_journey_archetype_summary_v2",
    "dev_hmm_business_signals_v2",
]

hmm_all_tables = hmm_v1_tables + hmm_v2_tables
hmm_table_files = {table: f"{table}.parquet" for table in hmm_all_tables}

print("HMM parquet folder:", HMM_PARQUET_DIR.resolve())
parquet_files = sorted(HMM_PARQUET_DIR.glob("dev_hmm_*.parquet"))
if parquet_files:
    for f in parquet_files:
        print(f.name)
else:
    print("No dev_hmm_*.parquet files found yet — copy exports into toexport_HMM/.")

HMM parquet folder: /Users/pmazolew/Documents/GitHub/Spring2026_IndustryProject/toexport_HMM
dev_hmm_developer_journey_v1.parquet
dev_hmm_run_stats_v1.parquet
dev_hmm_state_profile_v1.parquet
dev_hmm_transition_matrix_v1.parquet
dev_hmm_weekly_states_v1.parquet


In [13]:
for table in hmm_all_tables:
    con.execute(f"DROP TABLE IF EXISTS {table}")

print("Dropped existing HMM v1/v2 tables (if any).")

Dropped existing HMM v1/v2 tables (if any).


In [14]:
loaded_hmm = []
skipped_hmm = []

for table_name, file_name in hmm_table_files.items():
    file_path = HMM_PARQUET_DIR / file_name

    if file_path.exists():
        print(f"Loading {file_name} into {table_name}...")
        con.execute(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *
            FROM read_parquet('{file_path.as_posix()}')
        """)
        loaded_hmm.append(table_name)
    else:
        print(f"Skipping missing file: {file_name}")
        skipped_hmm.append(file_name)

if loaded_hmm:
    print(f"\nFinished loading {len(loaded_hmm)} HMM table(s) into DuckDB.")
else:
    print("\nNo HMM parquet files loaded. Add files to toexport_HMM/ and re-run.")

Loading dev_hmm_weekly_states_v1.parquet into dev_hmm_weekly_states_v1...
Loading dev_hmm_state_profile_v1.parquet into dev_hmm_state_profile_v1...
Loading dev_hmm_developer_journey_v1.parquet into dev_hmm_developer_journey_v1...
Loading dev_hmm_transition_matrix_v1.parquet into dev_hmm_transition_matrix_v1...
Loading dev_hmm_run_stats_v1.parquet into dev_hmm_run_stats_v1...
Skipping missing file: dev_hmm_state_labels_v2.parquet
Skipping missing file: dev_hmm_hdbscan_alignment_v2.parquet
Skipping missing file: dev_hmm_lifecycle_signal_summary_v2.parquet
Skipping missing file: dev_hmm_hdbscan_alignment_summary_v2.parquet
Skipping missing file: dev_hmm_journey_archetype_summary_v2.parquet
Skipping missing file: dev_hmm_business_signals_v2.parquet

Finished loading 5 HMM table(s) into DuckDB.


In [15]:
print("--- HMM v1 (base model outputs) ---")
for table in hmm_v1_tables:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {n:,} rows")
    except Exception:
        print(f"{table}: not loaded")

print("\n--- HMM v2 (journey analysis, optional) ---")
for table in hmm_v2_tables:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"{table}: {n:,} rows")
    except Exception:
        print(f"{table}: not loaded")

--- HMM v1 (base model outputs) ---
dev_hmm_weekly_states_v1: 916,268 rows
dev_hmm_state_profile_v1: 8 rows
dev_hmm_developer_journey_v1: 150,000 rows
dev_hmm_transition_matrix_v1: 64 rows
dev_hmm_run_stats_v1: 1 rows

--- HMM v2 (journey analysis, optional) ---
dev_hmm_state_labels_v2: not loaded
dev_hmm_hdbscan_alignment_v2: not loaded
dev_hmm_lifecycle_signal_summary_v2: not loaded
dev_hmm_hdbscan_alignment_summary_v2: not loaded
dev_hmm_journey_archetype_summary_v2: not loaded
dev_hmm_business_signals_v2: not loaded


In [16]:
# Reuses `con` from the setup cell — do not open a second connection to the same .duckdb file.
out_dir = Path("outputs/colab_hmm_inputs")
out_dir.mkdir(parents=True, exist_ok=True)

con.execute(f"""
COPY dev_gmm_weekly_clusters_v1
TO '{(out_dir / "dev_gmm_weekly_clusters_v1.parquet").as_posix()}'
(FORMAT PARQUET)
""")

con.execute(f"""
COPY dev_lifecycle_cluster_membership_v11_final
TO '{(out_dir / "dev_lifecycle_cluster_membership_v11_final.parquet").as_posix()}'
(FORMAT PARQUET)
""")

print("Exported HMM input parquet files.")
print("Output directory:", out_dir.resolve())

Exported HMM input parquet files.
Output directory: /Users/pmazolew/Documents/GitHub/Spring2026_IndustryProject/outputs/colab_hmm_inputs


In [17]:
out_path = Path("dev_weekly_features_v2.parquet")  # or outputs/v11_cluster_exports/

con.execute(f"""
COPY dev_weekly_features_v2
TO '{out_path.as_posix()}'
(FORMAT PARQUET);
""")

print("Wrote:", out_path.resolve())

Wrote: /Users/pmazolew/Documents/GitHub/Spring2026_IndustryProject/dev_weekly_features_v2.parquet


In [18]:
con.close()